In [15]:
import pandas as pd
import praw
import os
from dotenv import load_dotenv
import re
import spacy
import pickle
from tensorflow.keras.models import load_model
from wordcloud import WordCloud

In [16]:
load_dotenv()

reddit = praw.Reddit(
    client_id=os.getenv("CLIENT_ID"),
    client_secret=os.getenv("TOKEN_REDDIT"),
    user_agent=('app.app')
)

# Test de connexion
print(reddit.read_only)

True


In [8]:
subreddit = reddit.subreddit('movies')

commentaires = []

# Récupère les 100 posts les plus récents
for post in subreddit.hot(limit=100):
    post.comments.replace_more(limit=0)  # charge tous les commentaires
    for comment in post.comments.list():
        commentaires.append({
            'post_title': post.title,
            'comment'   : comment.body,
            'score'     : comment.score,
            'date'      : comment.created_utc
        })


df_reddit = pd.DataFrame(commentaires)
print(f"{len(df_reddit):,} commentaires récupérés")
df_reddit.head()
df_reddit.to_csv('./reddit_comments_test_Final.csv', index=False)

12,373 commentaires récupérés


In [17]:
df_reddit = pd.read_csv("./reddit_comments_test_Final.csv")

In [18]:
df_reddit

,post_title,comment,score,date
0,AMA/Q&A Announcement - Fisher Stevens - Friday...,"Fisher Stevens, actor & filmmaker, will be joi...",1,1.776444e+09
1,AMA/Q&A Announcement - Fisher Stevens - Friday...,Don't forget Hackers!\n\nHack the planet!,390,1.776444e+09
2,AMA/Q&A Announcement - Fisher Stevens - Friday...,Did you ever fix that review of Paddy's Pub?,179,1.776445e+09
3,AMA/Q&A Announcement - Fisher Stevens - Friday...,Does anyone at all happen to be old enough to ...,77,1.776446e+09
4,AMA/Q&A Announcement - Fisher Stevens - Friday...,Mr. The Plague!,67,1.776444e+09
...,...,...,...,...
12368,'Spaceballs 2' Gets Official Title - 'Spacebal...,don’t bother,1,1.776351e+09
12369,'Spaceballs 2' Gets Official Title - 'Spacebal...,"I hate to be that guy, but it would be 4 1/2",6,1.776352e+09
12370,'Spaceballs 2' Gets Official Title - 'Spacebal...,It looks to me like I’ve got all the Schwartze...,6,1.776339e+09
12371,'Spaceballs 2' Gets Official Title - 'Spacebal...,Actually now that I think about it it would be...,11,1.776353e+09


In [19]:
def clean_reddit(text):
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)         # URLs
    text = re.sub(r'\[.*?\]\(.*?\)', '', text)           # liens markdown [texte](url)
    text = re.sub(r'\*\*|__|\*|_|~~|`', '', text)        # formatage markdown
    text = re.sub(r'&gt;.*?\n', '', text)                # citations Reddit (>)
    text = re.sub(r'/r/\w+|/u/\w+', '', text)            # mentions subreddit/user
    text = re.sub(r'\n+', ' ', text)                     # sauts de ligne
    text = re.sub(r'[^a-zA-Z\s]', '', text)              # caractères spéciaux
    text = re.sub(r'\s+', ' ', text).strip().lower()     # espaces multiples
    return text
df_reddit['comment_clean'] = df_reddit['comment'].apply(clean_reddit)

### NLP

In [20]:
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

In [21]:
processed = []
for doc in nlp.pipe(df_reddit['comment_clean'].tolist(), batch_size=500, n_process=-1):
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and token.is_alpha
        and len(token.text) > 1
    ]
    processed.append(' '.join(tokens))

df_reddit['comment_nlp'] = processed
df_reddit = df_reddit[df_reddit['comment_nlp'].str.len() > 0].reset_index(drop=True)
print(f"Après NLP : {len(df_reddit):,} commentaires")

Après NLP : 12,233 commentaires


In [24]:
with open('../../notebook/entrainement/vectorizer_final.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

In [25]:
model = load_model("../../model/keras/Final.keras")

In [26]:
X_reddit = vectorizer(df_reddit['comment_nlp'].values)
predictions = model.predict(X_reddit)

383/383 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step


In [27]:
df_reddit['score_sentiment'] = predictions
df_reddit['sentiment']       = (predictions > 0.5).astype(int)
df_reddit['sentiment_label'] = df_reddit['sentiment'].map({0: 'Négatif', 1: 'Positif'})

print(df_reddit[['comment', 'sentiment_label', 'score_sentiment']].head(10))

                                             comment sentiment_label  \
0  Fisher Stevens, actor & filmmaker, will be joi...         Positif   
1          Don't forget Hackers!\n\nHack the planet!         Positif   
2       Did you ever fix that review of Paddy's Pub?         Positif   
3  Does anyone at all happen to be old enough to ...         Positif   
4                                    Mr. The Plague!         Négatif   
5             How the hell do you leave off Hackers?         Négatif   
6              Don't forget  succession- "woof woof"         Positif   
7  I always remember him as the filmmaker Columbo...         Positif   
8                 Don't forget 'My Science Project'          Positif   
9  Let's not forget his memorable role as Roger t...         Positif   

   score_sentiment  
0         0.748612  
1         0.823871  
2         0.682252  
3         0.572050  
4         0.072924  
5         0.232044  
6         0.985351  
7         0.705877  
8         0.777993

In [28]:
# Analyse des tendances
print("=== TENDANCES GLOBALES ===")
print(df_reddit['sentiment_label'].value_counts())
print()
print(df_reddit['sentiment_label'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

=== TENDANCES GLOBALES ===
sentiment_label
Positif    8674
Négatif    3559
Name: count, dtype: int64

sentiment_label
Positif    70.9%
Négatif    29.1%
Name: proportion, dtype: str


In [30]:
df_reddit.to_csv("./clean_nlp_comment_reddi_final_predict.csv")

In [32]:
df_reddit

,post_title,comment,score,date,comment_clean,comment_nlp,score_sentiment,sentiment,sentiment_label
0,AMA/Q&A Announcement - Fisher Stevens - Friday...,"Fisher Stevens, actor & filmmaker, will be joi...",1,1.776444e+09,fisher stevens actor filmmaker will be joining...,fisher steven actor filmmaker join amaqa frida...,0.748612,1,Positif
1,AMA/Q&A Announcement - Fisher Stevens - Friday...,Don't forget Hackers!\n\nHack the planet!,390,1.776444e+09,dont forget hackers hack the planet,not forget hacker hack planet,0.823871,1,Positif
2,AMA/Q&A Announcement - Fisher Stevens - Friday...,Did you ever fix that review of Paddy's Pub?,179,1.776445e+09,did you ever fix that review of paddys pub,fix review paddys pub,0.682252,1,Positif
3,AMA/Q&A Announcement - Fisher Stevens - Friday...,Does anyone at all happen to be old enough to ...,77,1.776446e+09,does anyone at all happen to be old enough to ...,happen old remember early edition guy get news...,0.572050,1,Positif
4,AMA/Q&A Announcement - Fisher Stevens - Friday...,Mr. The Plague!,67,1.776444e+09,mr the plague,mr plague,0.072924,0,Négatif
...,...,...,...,...,...,...,...,...,...
12228,'Spaceballs 2' Gets Official Title - 'Spacebal...,don’t bother,1,1.776351e+09,dont bother,not bother,0.601610,1,Positif
12229,'Spaceballs 2' Gets Official Title - 'Spacebal...,"I hate to be that guy, but it would be 4 1/2",6,1.776352e+09,i hate to be that guy but it would be,hate guy,0.109542,0,Négatif
12230,'Spaceballs 2' Gets Official Title - 'Spacebal...,It looks to me like I’ve got all the Schwartze...,6,1.776339e+09,it looks to me like ive got all the schwartzes,look like ve get schwartze,0.835434,1,Positif
12231,'Spaceballs 2' Gets Official Title - 'Spacebal...,Actually now that I think about it it would be...,11,1.776353e+09,actually now that i think about it it would be...,actually think total life half life cat life h...,0.677692,1,Positif


In [37]:
from IPython.display import display, HTML

with open('analyse_predictions_reddit.html', 'r') as f:
    html_content = f.read()

display(HTML(html_content))